# Visualize Exported Conversations

Browse exported conversation JSON files produced by the InSight-doc evaluation / VERL export path.

This notebook supports the shared export schema used by `insight_qwen_agent`, `insight_qwen_agent_core`, and VReasoner-style exports. Set `EXPORT_DIR` to an `exported_conversations/` directory, then run the cells top to bottom.


In [ ]:
from pathlib import Path
import os
import sys

import pandas as pd
from IPython.display import Markdown, Pretty, display

ROOT = Path(os.environ.get("REPO_ROOT", Path.cwd())).resolve()
VERL_ROOT = Path(os.environ.get("VERL_ROOT") or ROOT / "verl").resolve()
for path in (ROOT, VERL_ROOT):
    if path.exists() and str(path) not in sys.path:
        sys.path.insert(0, str(path))

from verl.utils.vreasoner_v2_conversation_export import (
    load_exported_conversation,
    restore_conversation_for_visualization,
)

EXPORT_DIR = Path(os.environ.get("EXPORT_DIR", "/path/to/exported_conversations")).expanduser()
assert EXPORT_DIR.exists(), f"Set EXPORT_DIR to an exported_conversations directory: {EXPORT_DIR}"

export_paths = sorted(EXPORT_DIR.rglob("*.json"))
assert export_paths, f"No exported conversation JSON files found under: {EXPORT_DIR}"

records = [load_exported_conversation(str(path)) for path in export_paths]
print(f"Loaded {len(records)} exported conversations from {EXPORT_DIR}")


In [ ]:
def _score_field(record, key, default=None):
    reward = record.get("reward") or {}
    score = reward.get("score") or {}
    return score.get(key, default)


def _count_tool_calls(record):
    conversation = record.get("conversation") or []
    count = sum(1 for msg in conversation if msg.get("role") == "assistant" and msg.get("type") == "tool_call")
    if count:
        return count
    return _score_field(record, "n_valid_tool_calls")


rows = []
for idx, (path, record) in enumerate(zip(export_paths, records)):
    reward = record.get("reward") or {}
    extra_info = record.get("extra_info") or {}
    job = record.get("job") or {}
    status = record.get("status") or {}
    rows.append({
        "idx": idx,
        "file": path.name,
        "agent_name": record.get("agent_name"),
        "subset": extra_info.get("subset") or extra_info.get("data_source"),
        "question_id": extra_info.get("question_id"),
        "question_type": extra_info.get("question_type"),
        "question": extra_info.get("question"),
        "reward": reward.get("reward"),
        "score": _score_field(record, "score"),
        "accuracy_reward": _score_field(record, "accuracy_reward"),
        "format_reward": _score_field(record, "format_reward"),
        "tool_calls": _count_tool_calls(record),
        "critical_failure": status.get("critical_failure"),
        "extracted_answer": reward.get("extracted_answer"),
        "ground_truth": reward.get("ground_truth"),
        "job_id": job.get("job_id") or job.get("request_id"),
    })

summary_df = pd.DataFrame(rows)
summary_df.head(20)


In [ ]:
metric_cols = ["reward", "score", "accuracy_reward", "format_reward", "tool_calls"]
existing_metric_cols = [col for col in metric_cols if col in summary_df]

group_cols = [col for col in ["agent_name", "subset"] if col in summary_df]
if group_cols:
    aggregate_df = (
        summary_df.groupby(group_cols, dropna=False)
        .agg(
            n=("idx", "count"),
            reward=("reward", "mean"),
            accuracy_reward=("accuracy_reward", "mean"),
            tool_calls=("tool_calls", "mean"),
        )
        .reset_index()
        .sort_values(group_cols)
    )
    display(aggregate_df)

if "tool_calls" in summary_df:
    display(summary_df["tool_calls"].value_counts(dropna=False).sort_index().rename_axis("tool_calls").reset_index(name="count"))

summary_df[existing_metric_cols].describe()


In [ ]:
OMIT_SYSTEM_PROMPT = False
MAX_TEXT_CHARS = 20_000


def _short_text(text, max_chars=MAX_TEXT_CHARS):
    text = "" if text is None else str(text)
    if len(text) <= max_chars:
        return text
    return text[:max_chars] + f"\n\n[truncated: {len(text) - max_chars} chars omitted]"

def display_restored_conversation(restored_payload, *, omit_system_prompt=OMIT_SYSTEM_PROMPT):
    images = list(restored_payload.get("multi_modal_data", {}).get("images", []))
    image_idx = 0
    agent_name = restored_payload.get("record", {}).get("agent_name", "unknown-agent")
    display(Markdown(f"`agent_name = {agent_name}`"))

    for message in restored_payload.get("messages", []):
        role = message.get("role", "unknown")
        display(Markdown(f"**{role.capitalize()}:**"))

        if omit_system_prompt and role == "system":
            display(Pretty("[system prompt omitted]"))
            continue

        contents = message.get("content", "")
        if not isinstance(contents, list):
            contents = [contents]

        for content in contents:
            if isinstance(content, str):
                display(Pretty(_short_text(content)))
                continue

            kind = content.get("type")
            if kind == "text":
                text = content.get("text", "")
                if text:
                    display(Pretty(_short_text(text)))
            elif kind == "image":
                image = images[image_idx] if image_idx < len(images) else None
                if image is None:
                    display(Pretty("[image unavailable from reference]"))
                else:
                    display(image)
                    print("image size:", image.size)
                image_idx += 1
            else:
                display(Pretty(content))


def display_conversation(idx, *, omit_system_prompt=OMIT_SYSTEM_PROMPT, show_image_table=True):
    idx = int(idx)
    path = export_paths[idx]
    record = records[idx]
    restored = restore_conversation_for_visualization(record)

    display(Markdown(f"### Row {idx}: `{path.name}`"))
    display(summary_df.loc[[idx]].T)

    reward = record.get("reward") or {}
    if reward:
        display(Markdown("**Reward / judge payload**"))
        display(pd.json_normalize(reward, sep=".").T)

    if show_image_table:
        image_rows = []
        for item in restored.get("presented_images", []):
            image_rows.append({
                "presented_img_idx": item.get("presented_img_idx"),
                "kind": item.get("kind"),
                "source_original_img_idx": item.get("source_original_img_idx"),
                "parent_presented_img_idx": item.get("parent_presented_img_idx"),
                "bbox_on_original": item.get("bbox_on_original"),
                "display_size": item.get("display_size"),
                "region_description": item.get("region_description"),
                "image_restored": item.get("image") is not None,
            })
        if image_rows:
            display(Markdown("**Presented images**"))
            display(pd.DataFrame(image_rows))

    print("EXPORT_PATH:", path)
    display_restored_conversation(restored, omit_system_prompt=omit_system_prompt)
    return restored


In [ ]:
# Pick a row index from summary_df.
idx = 0
restored = display_conversation(idx)


In [ ]:
# Optional filters. Uncomment or edit as needed.

# Failed / imperfect examples:
# summary_df[summary_df["reward"] != 1.0].head(20)

# Many tool calls:
# summary_df[summary_df["tool_calls"].fillna(0) >= 5].head(20)

# Critical failures:
# summary_df[summary_df["critical_failure"].fillna(False)].head(20)

# Search by question text:
# summary_df[summary_df["question"].fillna("").str.contains("invoice", case=False)]
